In [ ]:
import platform
import shutil
import subprocess
import sys

print("Operating system:", platform.system())
print("Python:", sys.version.split()[0])
print("Python location:", sys.executable)

if platform.system() == "Darwin":
    print("\nSTOP: this kernel is running locally on your Mac.")
elif shutil.which("nvidia-smi"):
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free,driver_version",
            "--format=csv",
        ],
        capture_output=True,
        text=True,
    )
    print("\nGPU information:")
    print(result.stdout or result.stderr)
else:
    print("\nNo NVIDIA GPU detected in this runtime.")
    print("Connect the notebook to a GPU runtime before continuing.")

# CSEV isolated GPU validation
Select an **A100 GPU** runtime, then run these cells in order. Your Mac is only the browser; computation runs on Colab.

Upload the bundle created by `fetch_and_pack.sh`. This tests 15 collected receipts and all 135 saved attacks (three attack families × three budgets), with all nine defense conditions: 1,350 condition records. It also tests an adaptive gradient update and replay on three receipts. This is substantially larger than a quick smoke test and may take hours; progress is printed after every image.

Uses the CARC source commit, model revision, precision, resolution settings and recorded Python package versions. Uses the same OCR language data, but Colab's OCR binary, drivers, OS and GPU differ. No CARC connection or job changes occur. A diagnostic threshold of zero is **not calibrated**; do not put these metrics in the report as final results. Full adaptive search, calibration and Slurm orchestration are not validated here.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile,tarfile,json,subprocess,sys,os
uploaded=files.upload()
assert 'csev_colab_bundle.zip' in uploaded, 'Upload csev_colab_bundle.zip'
ROOT=Path('/content/csev_validation')
assert not ROOT.exists(), 'Directory already exists; use a fresh runtime to avoid mixing runs.'
ROOT.mkdir()
with zipfile.ZipFile('csev_colab_bundle.zip') as z:
    for info in z.infolist():
        assert (ROOT/info.filename).resolve().is_relative_to(ROOT)
    z.extractall(ROOT)
(ROOT/'source').mkdir()
with tarfile.open(ROOT/'source.tar') as t:
    for member in t.getmembers():
        assert (ROOT/'source'/member.name).resolve().is_relative_to(ROOT/'source')
        assert not member.issym() and not member.islnk()
    t.extractall(ROOT/'source')
config=json.loads((ROOT/'csev_receipts_v1.json').read_text())
print('Source commit:',config['git_commit'])
subprocess.run(['nvidia-smi'],check=True)


## Install the recorded environment
This creates a separate Python environment using [uv](https://docs.astral.sh/uv/concepts/python-versions/). It does not rely on Colab's preinstalled PyTorch. If an exact version is unavailable, stop and share the error; do not silently change package versions. Model weights download separately from Hugging Face on first use.

In [ ]:
def run(args):
    print(' '.join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),check=True)
run([sys.executable,'-m','pip','install','uv'])
run(['uv','venv','--python',config['runtime_versions']['python'],'--seed',ROOT/'venv'])
PYTHON=ROOT/'venv/bin/python'
versions=config['runtime_versions']
run(['uv','pip','install','--python',PYTHON,'--index-url','https://download.pytorch.org/whl/cu124',
     'torch=='+versions['torch'],'torchvision=='+versions['torchvision']])
packages=[k+'=='+v for k,v in versions.items() if k not in ('python','torch','torchvision')]
run(['uv','pip','install','--python',PYTHON,*packages,'pyarrow','sentencepiece','accelerate'])
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','tesseract-ocr'])
run([PYTHON,'-m','pip','freeze'])


## Run validation
Do not reduce image resolution or change BF16 to make an out-of-memory error disappear: that would change the tested workload. A rejection by the defense is a normal outcome; an exception is a runtime failure. The first run downloads the Qwen checkpoint. All per-image results are saved immediately. Download partial results after a failure using the last cell.

In [ ]:
env=os.environ.copy()
env['HF_HOME']=str(ROOT/'hf_cache')
with (ROOT/'validation.log').open('w') as log:
    process=subprocess.Popen([str(PYTHON),'-u',str(ROOT/'validate_gpu.py')],cwd=ROOT,env=env,
                             stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='');log.write(line);log.flush()
    returncode=process.wait()
print('Exit code:',returncode)
print('Read summary.json if successful, or failure.json if it failed.')


In [ ]:
import shutil
export=ROOT/'export';export.mkdir(exist_ok=True)
if (ROOT/'validation_results').exists():
    shutil.copytree(ROOT/'validation_results',export/'validation_results',dirs_exist_ok=True)
if (ROOT/'validation.log').exists():shutil.copy2(ROOT/'validation.log',export/'validation.log')
shutil.copy2(ROOT/'csev_receipts_v1.json',export/'original_carc_config.json')
archive=shutil.make_archive('/content/csev_validation_results','zip',export)
files.download(archive)
